# Energy Forecasting — Data Exploration (2023–2024 Individual Datasets)

This notebook explores the three individual `2023–2024` raw datasets in `Dataset/raw/` (`Actual Consumption`, `Day-Ahead Prices`, and `Open-Meteo Weather`) for data types and Non-Numeric Placeholder Tokens NAs, outliers, domain-specific impossible values with checking and interpolating, duplicate timestamps / DST transitions, and target load statistics.

## 1. Setup & File Paths

In [34]:
import locale
from pathlib import Path

import pandas as pd
from IPython.display import display

# Resolve repository root and Dataset/raw directory
PROJECT_ROOT = Path.cwd().resolve()
for candidate in (PROJECT_ROOT, *PROJECT_ROOT.parents):
    if (candidate / "Dataset").is_dir():
        PROJECT_ROOT = candidate
        break

DATASET_DIR = PROJECT_ROOT / "Dataset"
RAW_DIR = DATASET_DIR / "raw" / "xlsx"

CONSUMPTION_FILE = RAW_DIR / "Actual_consumption_2023_2024_Quarterhour.xlsx"
PRICES_FILE = RAW_DIR / "Day-ahead_prices_2023_2024_Quarterhour.xlsx"
WEATHER_FILE = RAW_DIR / "open-meteo-48.83N9.15E293m_2023_2024.xlsx"

# Key columns defined in Dataset/README.md
TARGET_COL = "grid load [MWh]"
PRICE_COL = "Germany/Luxembourg [€/MWh]"
WEATHER_COLS = [
    "temperature_2m (°C)",
    "relative_humidity_2m (%)",
    "precipitation (mm)",
    "shortwave_radiation (W/m²)",
    "wind_speed_10m (km/h)",
    "dew_point_2m (°C)",
    "weather_code (wmo code)",
]

print("Raw directory:     ", RAW_DIR)
print("Consumption file:  ", CONSUMPTION_FILE.name)
print("Prices file:       ", PRICES_FILE.name)
print("Weather file:      ", WEATHER_FILE.name)

Raw directory:      C:\Users\Win11\Documents\GitHub\Energy-Forecasting\Dataset\raw
Consumption file:   Actual_consumption_2023_2024_Quarterhour.xlsx
Prices file:        Day-ahead_prices_2023_2024_Quarterhour.xlsx
Weather file:       open-meteo-48.83N9.15E293m_2023_2024.xlsx


## 2. Load 2023–2024 Raw Datasets

In [35]:
raw_consumption = pd.read_excel(CONSUMPTION_FILE, skiprows=9)
raw_prices = pd.read_excel(PRICES_FILE, skiprows=9)
raw_weather = pd.read_excel(WEATHER_FILE, skiprows=3)

datasets = {
    "Actual Consumption": raw_consumption,
    "Day-Ahead Prices": raw_prices,
    "Open-Meteo Weather": raw_weather,
}

for name, df in datasets.items():
    print(f"{name}: {df.shape[0]:,} rows x {df.shape[1]} columns")

Actual Consumption: 70,176 rows x 6 columns
Day-Ahead Prices: 70,176 rows x 19 columns
Open-Meteo Weather: 70,176 rows x 10 columns


In [36]:
print("Consumption (2023–2024)")
display(raw_consumption.head(3))

print("Energy Prices (2023–2024)")
display(raw_prices.head(3))

print("Weather Data (2023–2024)")
display(raw_weather.head(3))

Consumption (2023–2024)


,Start date,End date,grid load [MWh],Grid load incl. hydro pumped storage [MWh],Hydro pumped storage [MWh],Residual load [MWh]
0,"Jan 1, 2023 12:00 AM","Jan 1, 2023 12:15 AM",1122.25,1366.25,244.00,986.75
1,"Jan 1, 2023 12:15 AM","Jan 1, 2023 12:30 AM",1107.00,1341.25,234.25,962.5
2,"Jan 1, 2023 12:30 AM","Jan 1, 2023 12:45 AM",1109.75,1414.00,304.25,943.25


Energy Prices (2023–2024)


,Start date,End date,Germany/Luxembourg [€/MWh],∅ DE/LU neighbours [€/MWh],Belgium [€/MWh],Denmark 1 [€/MWh],Denmark 2 [€/MWh],France [€/MWh],Netherlands [€/MWh],Norway 2 [€/MWh],Austria [€/MWh],Poland [€/MWh],Sweden 4 [€/MWh],Switzerland [€/MWh],Czech Republic [€/MWh],DE/AT/LU [€/MWh],Northern Italy [€/MWh],Slovenia [€/MWh],Hungary [€/MWh]
0,"Jan 1, 2023 12:00 AM","Jan 1, 2023 12:15 AM",-5.17,13.85,-4.39,2.01,2.01,0.0,-3.61,119.32,12.06,18.09,2.01,0.03,4.84,-,195.9,13.31,19.76
1,"Jan 1, 2023 12:15 AM","Jan 1, 2023 12:30 AM",-5.17,13.85,-4.39,2.01,2.01,0.0,-3.61,119.32,12.06,18.09,2.01,0.03,4.84,-,195.9,13.31,19.76
2,"Jan 1, 2023 12:30 AM","Jan 1, 2023 12:45 AM",-5.17,13.85,-4.39,2.01,2.01,0.0,-3.61,119.32,12.06,18.09,2.01,0.03,4.84,-,195.9,13.31,19.76


Weather Data (2023–2024)


,time,temperature_2m (°C),relative_humidity_2m (%),precipitation (mm),shortwave_radiation (W/m²),direct_radiation (W/m²),sunshine_duration (s),wind_speed_10m (km/h),weather_code (wmo code),dew_point_2m (°C)
0,2023-01-01 00:00:00,13.0,75,0.0,0.0,0,0.0,6.3,3,8.7
1,2023-01-01 00:15:00,12.9,75,0.0,0.0,0,0.0,6.6,2,8.6
2,2023-01-01 00:30:00,12.9,74,0.0,0.0,0,0.0,7.8,2,8.4


## 3. Data Types & Non-Numeric Placeholder Tokens
Inspect column `dtypes`, count non-numeric tokens (such as `'-'` or `'#NUM!'`), and convert timestamps to `datetime` and value columns to `float64` for each individual dataset.

In [37]:
def parse_timestamp_series(series: pd.Series) -> pd.Series:
    if pd.api.types.is_datetime64_any_dtype(series):
        return series
    if pd.api.types.is_numeric_dtype(series):
        return pd.to_datetime(series, unit="D", origin="1899-12-30").dt.round("min")
    # Use C locale so English month names and AM/PM parse on German Windows
    locale.setlocale(locale.LC_TIME, "C")
    return pd.to_datetime(series, format="%b %d, %Y %I:%M %p", errors="coerce")


timestamp_config = {
    "Actual Consumption": (["Start date", "End date"], "Start date"),
    "Day-Ahead Prices": (["Start date", "End date"], "Start date"),
    "Open-Meteo Weather": (["time"], "time"),
}

coerced_datasets = {}

for name, df in datasets.items():
    ts_cols, _ = timestamp_config[name]
    coerced_df = df.copy()
    type_rows = []

    for col in df.columns:
        raw_dtype = str(df[col].dtype)
        if col in ts_cols:
            parsed_ts = parse_timestamp_series(df[col])
            invalid_count = int(parsed_ts.isna().sum() - df[col].isna().sum())
            coerced_df[col] = parsed_ts
            type_rows.append({
                "column": col,
                "raw_dtype": raw_dtype,
                "expected_type": "datetime",
                "non_numeric_or_invalid": invalid_count,
                "sample_invalid_tokens": "",
            })
        else:
            s = df[col]
            numeric_s = pd.to_numeric(s, errors="coerce")
            non_numeric_mask = s.notna() & numeric_s.isna()
            non_numeric_count = int(non_numeric_mask.sum())
            sample_tokens = (
                s[non_numeric_mask].astype(str).value_counts().head(3).to_dict()
                if non_numeric_count > 0
                else "None"
            )
            coerced_df[col] = numeric_s
            type_rows.append({
                "column": col,
                "raw_dtype": raw_dtype,
                "expected_type": "float64",
                "non_numeric_or_invalid": non_numeric_count,
                "sample_invalid_tokens": str(sample_tokens),
            })

    coerced_datasets[name] = coerced_df
    print(f"=== {name} ===")
    display(pd.DataFrame(type_rows))

=== Actual Consumption ===


,column,raw_dtype,expected_type,non_numeric_or_invalid,sample_invalid_tokens
0,Start date,str,datetime,0,
1,End date,str,datetime,0,
2,grid load [MWh],float64,float64,0,None
3,Grid load incl. hydro pumped storage [MWh],float64,float64,0,None
4,Hydro pumped storage [MWh],float64,float64,0,None
5,Residual load [MWh],object,float64,1,{'-': 1}


=== Day-Ahead Prices ===


,column,raw_dtype,expected_type,non_numeric_or_invalid,sample_invalid_tokens
0,Start date,str,datetime,0,
1,End date,str,datetime,0,
2,Germany/Luxembourg [€/MWh],float64,float64,0,None
3,∅ DE/LU neighbours [€/MWh],float64,float64,0,None
4,Belgium [€/MWh],float64,float64,0,None
5,Denmark 1 [€/MWh],float64,float64,0,None
6,Denmark 2 [€/MWh],float64,float64,0,None
7,France [€/MWh],float64,float64,0,None
8,Netherlands [€/MWh],float64,float64,0,None
9,Norway 2 [€/MWh],float64,float64,0,None


=== Open-Meteo Weather ===


,column,raw_dtype,expected_type,non_numeric_or_invalid,sample_invalid_tokens
0,time,datetime64[us],datetime,0,
1,temperature_2m (°C),float64,float64,0,None
2,relative_humidity_2m (%),int64,float64,0,None
3,precipitation (mm),float64,float64,0,None
4,shortwave_radiation (W/m²),float64,float64,0,None
5,direct_radiation (W/m²),int64,float64,0,None
6,sunshine_duration (s),float64,float64,0,None
7,wind_speed_10m (km/h),float64,float64,0,None
8,weather_code (wmo code),int64,float64,0,None
9,dew_point_2m (°C),float64,float64,0,None


## 4. Missing Data Analysis (Individual Datasets)

In [38]:
for name, coerced_df in coerced_datasets.items():
    raw_df = datasets[name]
    _, primary_ts = timestamp_config[name]
    ts = coerced_df[primary_ts]
    total_rows = len(coerced_df)

    missing_rows = []
    for col in coerced_df.columns:
        raw_na = int(raw_df[col].isna().sum())
        coerced_na = int(coerced_df[col].isna().sum())
        valid_mask = coerced_df[col].notna()

        missing_rows.append({
            "column": col,
            "raw_nulls": raw_na,
            "placeholder_nulls": coerced_na - raw_na,
            "total_missing": coerced_na,
            "missing_%": round((coerced_na / total_rows) * 100.0, 2),
            "first_valid_ts": ts[valid_mask].min() if valid_mask.any() else None,
            "last_valid_ts": ts[valid_mask].max() if valid_mask.any() else None,
        })

    print(f"=== {name} ===")
    display(pd.DataFrame(missing_rows))

=== Actual Consumption ===


,column,raw_nulls,placeholder_nulls,total_missing,missing_%,first_valid_ts,last_valid_ts
0,Start date,0,0,0,0.0,2023-01-01,2024-12-31 23:45:00
1,End date,0,0,0,0.0,2023-01-01,2024-12-31 23:45:00
2,grid load [MWh],0,0,0,0.0,2023-01-01,2024-12-31 23:45:00
3,Grid load incl. hydro pumped storage [MWh],0,0,0,0.0,2023-01-01,2024-12-31 23:45:00
4,Hydro pumped storage [MWh],0,0,0,0.0,2023-01-01,2024-12-31 23:45:00
5,Residual load [MWh],0,1,1,0.0,2023-01-01,2024-12-31 23:45:00


=== Day-Ahead Prices ===


,column,raw_nulls,placeholder_nulls,total_missing,missing_%,first_valid_ts,last_valid_ts
0,Start date,0,0,0,0.0,2023-01-01,2024-12-31 23:45:00
1,End date,0,0,0,0.0,2023-01-01,2024-12-31 23:45:00
2,Germany/Luxembourg [€/MWh],0,0,0,0.0,2023-01-01,2024-12-31 23:45:00
3,∅ DE/LU neighbours [€/MWh],0,0,0,0.0,2023-01-01,2024-12-31 23:45:00
4,Belgium [€/MWh],0,0,0,0.0,2023-01-01,2024-12-31 23:45:00
5,Denmark 1 [€/MWh],0,0,0,0.0,2023-01-01,2024-12-31 23:45:00
6,Denmark 2 [€/MWh],0,0,0,0.0,2023-01-01,2024-12-31 23:45:00
7,France [€/MWh],0,0,0,0.0,2023-01-01,2024-12-31 23:45:00
8,Netherlands [€/MWh],0,0,0,0.0,2023-01-01,2024-12-31 23:45:00
9,Norway 2 [€/MWh],0,0,0,0.0,2023-01-01,2024-12-31 23:45:00


=== Open-Meteo Weather ===


,column,raw_nulls,placeholder_nulls,total_missing,missing_%,first_valid_ts,last_valid_ts
0,time,0,0,0,0.0,2023-01-01,2024-12-31 23:45:00
1,temperature_2m (°C),0,0,0,0.0,2023-01-01,2024-12-31 23:45:00
2,relative_humidity_2m (%),0,0,0,0.0,2023-01-01,2024-12-31 23:45:00
3,precipitation (mm),0,0,0,0.0,2023-01-01,2024-12-31 23:45:00
4,shortwave_radiation (W/m²),0,0,0,0.0,2023-01-01,2024-12-31 23:45:00
5,direct_radiation (W/m²),0,0,0,0.0,2023-01-01,2024-12-31 23:45:00
6,sunshine_duration (s),0,0,0,0.0,2023-01-01,2024-12-31 23:45:00
7,wind_speed_10m (km/h),0,0,0,0.0,2023-01-01,2024-12-31 23:45:00
8,weather_code (wmo code),0,0,0,0.0,2023-01-01,2024-12-31 23:45:00
9,dew_point_2m (°C),0,0,0,0.0,2023-01-01,2024-12-31 23:45:00


## 5. Outliers & Domain-Specific Impossible Values (Individual Datasets)

In [39]:
for name, coerced_df in coerced_datasets.items():
    ts_cols, _ = timestamp_config[name]
    value_cols = [c for c in coerced_df.columns if c not in ts_cols]

    stats_rows = []
    for col in value_cols:
        s = coerced_df[col].dropna()
        if s.empty:
            continue
        q01, q25, q50, q75, q99 = s.quantile([0.01, 0.25, 0.50, 0.75, 0.99])
        iqr = q75 - q25
        std = s.std()

        stats_rows.append({
            "column": col,
            "min": round(float(s.min()), 2),
            "p01": round(float(q01), 2),
            "median": round(float(q50), 2),
            "mean": round(float(s.mean()), 2),
            "p99": round(float(q99), 2),
            "max": round(float(s.max()), 2),
            "iqr_1.5x": int(((s < q25 - 1.5 * iqr) | (s > q75 + 1.5 * iqr)).sum()),
            "iqr_3.0x": int(((s < q25 - 3.0 * iqr) | (s > q75 + 3.0 * iqr)).sum())
        })

    print(f"=== {name}: Distribution & Outlier Counts ===")
    display(pd.DataFrame(stats_rows))

    print(f"Domain-specific sanity checks ({name}):")
    if name == "Actual Consumption":
        for col in ["grid load [MWh]", "Grid load incl. hydro pumped storage [MWh]"]:
            print(f"  - {col} <= 0: {int((coerced_df[col] <= 0).sum()):,}")
        print(f"  - Hydro pumped storage [MWh] < 0: {int((coerced_df['Hydro pumped storage [MWh]'] < 0).sum()):,}")
    elif name == "Day-Ahead Prices":
        s = coerced_df[PRICE_COL].dropna()
        print(f"  - {PRICE_COL} outside SDAC caps (<-500 or >4000 €/MWh): {int(((s < -500) | (s > 4000)).sum()):,}")
    elif name == "Open-Meteo Weather":
        checks = [
            ("temperature_2m (°C)", lambda x: (x < -40) | (x > 50), "outside [-40, 50] °C"),
            ("dew_point_2m (°C)", lambda x: (x < -50) | (x > 45), "outside [-50, 45] °C"),
            ("relative_humidity_2m (%)", lambda x: (x < 0) | (x > 100), "outside [0, 100] %"),
            ("precipitation (mm)", lambda x: x < 0, "< 0 mm"),
            ("shortwave_radiation (W/m²)", lambda x: x < 0, "< 0 W/m²"),
            ("direct_radiation (W/m²)", lambda x: x < 0, "< 0 W/m²"),
            ("sunshine_duration (s)", lambda x: (x < 0) | (x > 900), "outside [0, 900] s"),
            ("wind_speed_10m (km/h)", lambda x: x < 0, "< 0 km/h"),
            ("weather_code (wmo code)", lambda x: (x < 0) | (x > 99), "outside [0, 99]"),
        ]
        for col, rule, desc in checks:
            s = coerced_df[col].dropna()
            print(f"  - {col} {desc}: {int(rule(s).sum()):,}")
        dew_gt_temp = int((coerced_df["dew_point_2m (°C)"] > coerced_df["temperature_2m (°C)"] + 1e-6).sum())
        print(f"  - dew_point_2m > temperature_2m (physically impossible): {dew_gt_temp:,}")
    print()

=== Actual Consumption: Distribution & Outlier Counts ===


,column,min,p01,median,mean,p99,max,iqr_1.5x,iqr_3.0x
0,grid load [MWh],872.75,988.50,1604.5,1624.89,2333.38,2498.00,0,0
1,Grid load incl. hydro pumped storage [MWh],890.25,1045.44,1698.0,1709.56,2367.00,2556.50,0,0
2,Hydro pumped storage [MWh],0.00,0.00,48.5,84.67,362.06,532.25,1118,0
3,Residual load [MWh],-472.00,45.44,1296.5,1290.80,2198.75,2388.25,1593,0


Domain-specific sanity checks (Actual Consumption):
  - grid load [MWh] <= 0: 0
  - Grid load incl. hydro pumped storage [MWh] <= 0: 0
  - Hydro pumped storage [MWh] < 0: 0

=== Day-Ahead Prices: Distribution & Outlier Counts ===


,column,min,p01,median,mean,p99,max,iqr_1.5x,iqr_3.0x
0,Germany/Luxembourg [€/MWh],-500.00,-11.46,88.87,86.83,214.16,936.28,2144,344
1,∅ DE/LU neighbours [€/MWh],-152.18,-3.41,81.12,82.27,193.45,664.26,2036,256
2,Belgium [€/MWh],-140.00,-12.89,85.77,83.78,202.10,565.46,1604,164
3,Denmark 1 [€/MWh],-440.10,-4.07,80.69,78.72,208.22,936.28,1172,220
4,Denmark 2 [€/MWh],-60.04,-2.12,76.59,76.05,217.25,936.31,924,176
5,France [€/MWh],-134.94,-5.32,80.38,77.41,195.83,284.21,612,0
6,Netherlands [€/MWh],-500.00,-25.34,89.60,86.54,208.07,872.96,2800,400
7,Norway 2 [€/MWh],-61.84,-1.53,59.34,64.77,166.53,898.25,1180,196
8,Austria [€/MWh],-500.00,-6.65,91.57,91.83,209.55,850.00,2132,320
9,Poland [€/MWh],-61.64,-0.81,101.93,103.94,234.64,630.19,4468,504


Domain-specific sanity checks (Day-Ahead Prices):
  - Germany/Luxembourg [€/MWh] outside SDAC caps (<-500 or >4000 €/MWh): 0

=== Open-Meteo Weather: Distribution & Outlier Counts ===


,column,min,p01,median,mean,p99,max,iqr_1.5x,iqr_3.0x
0,temperature_2m (°C),-7.5,-3.7,11.6,12.16,30.3,36.1,41,0
1,relative_humidity_2m (%),14.0,29.0,77.0,73.10,100.0,100.0,667,0
2,precipitation (mm),-0.1,0.0,0.0,0.02,0.5,19.3,5796,5796
3,shortwave_radiation (W/m²),0.0,0.0,2.0,131.09,801.0,930.0,6569,969
4,direct_radiation (W/m²),0.0,0.0,0.0,69.27,649.0,806.0,12513,9186
5,sunshine_duration (s),0.0,0.0,0.0,243.65,900.0,900.0,0,0
6,wind_speed_10m (km/h),0.0,0.8,5.9,7.33,23.0,33.0,1805,15
7,weather_code (wmo code),0.0,0.0,3.0,14.78,80.0,99.0,21863,14391
8,dew_point_2m (°C),-11.2,-7.0,7.0,6.94,18.8,22.3,58,0


Domain-specific sanity checks (Open-Meteo Weather):
  - temperature_2m (°C) outside [-40, 50] °C: 0
  - dew_point_2m (°C) outside [-50, 45] °C: 0
  - relative_humidity_2m (%) outside [0, 100] %: 0
  - precipitation (mm) < 0 mm: 1
  - shortwave_radiation (W/m²) < 0 W/m²: 0
  - direct_radiation (W/m²) < 0 W/m²: 0
  - sunshine_duration (s) outside [0, 900] s: 0
  - wind_speed_10m (km/h) < 0 km/h: 0
  - weather_code (wmo code) outside [0, 99]: 0
  - dew_point_2m > temperature_2m (physically impossible): 0



### 5.1 Inspect & Interpolate `precipitation (mm) < 0` in Open-Meteo Weather
Inspect the 5 rows above and 5 rows below `precipitation (mm) < 0` and interpolate the negative value as the mean of those 10 surrounding rows rounded to 1 decimal place.

In [40]:
weather_coerced = coerced_datasets["Open-Meteo Weather"]
neg_precip_indices = weather_coerced.index[weather_coerced["precipitation (mm)"] < 0].tolist()
print(f"Indices where precipitation (mm) < 0: {neg_precip_indices}")

for idx in neg_precip_indices:
    above_slice = weather_coerced.iloc[max(0, idx - 5) : idx]
    below_slice = weather_coerced.iloc[idx + 1 : min(len(weather_coerced), idx + 6)]
    surrounding_10 = pd.concat([above_slice, below_slice])

    print(f"\nWindow around index {idx} (5 rows above + target row + 5 rows below, before interpolation):")
    display(weather_coerced.iloc[max(0, idx - 5) : min(len(weather_coerced), idx + 6)])

    old_val = weather_coerced.loc[idx, "precipitation (mm)"]
    new_val = round(float(surrounding_10["precipitation (mm)"].mean()), 1)
    weather_coerced.loc[idx, "precipitation (mm)"] = new_val
    raw_weather.loc[idx, "precipitation (mm)"] = new_val

    print(
        f"Interpolated index {idx} ({weather_coerced.loc[idx, 'time']}): "
        f"{old_val} mm -> {new_val} mm "
        f"(mean of 5 above {above_slice['precipitation (mm)'].tolist()} "
        f"and 5 below {below_slice['precipitation (mm)'].tolist()}, rounded to 1 decimal place)"
    )

    print(f"\nWindow around index {idx} (after interpolation):")
    display(weather_coerced.iloc[max(0, idx - 5) : min(len(weather_coerced), idx + 6)])

Indices where precipitation (mm) < 0: [32075]

Window around index 32075 (5 rows above + target row + 5 rows below, before interpolation):


,time,temperature_2m (°C),relative_humidity_2m (%),precipitation (mm),shortwave_radiation (W/m²),direct_radiation (W/m²),sunshine_duration (s),wind_speed_10m (km/h),weather_code (wmo code),dew_point_2m (°C)
32070,2023-12-01 01:30:00,1.7,95,0.0,0.0,0,0.0,3.2,45,1.0
32071,2023-12-01 01:45:00,1.6,95,0.0,0.0,0,0.0,3.4,45,0.9
32072,2023-12-01 02:00:00,1.6,95,0.0,0.0,0,0.0,3.4,45,0.9
32073,2023-12-01 02:15:00,1.6,95,0.0,0.0,0,0.0,3.2,61,0.9
32074,2023-12-01 02:30:00,1.6,94,0.0,0.0,0,0.0,3.3,61,0.8
32075,2023-12-01 02:45:00,1.6,94,-0.1,0.0,0,0.0,3.1,61,0.8
32076,2023-12-01 03:00:00,1.5,94,0.2,0.0,0,0.0,3.2,61,0.7
32077,2023-12-01 03:15:00,1.3,95,0.1,0.0,0,0.0,3.2,71,0.6
32078,2023-12-01 03:30:00,1.0,96,0.2,0.0,0,0.0,2.9,71,0.4
32079,2023-12-01 03:45:00,0.6,97,0.3,0.0,0,0.0,2.8,71,0.2


Interpolated index 32075 (2023-12-01 02:45:00): -0.1 mm -> 0.1 mm (mean of 5 above [0.0, 0.0, 0.0, 0.0, 0.0] and 5 below [0.2, 0.1, 0.2, 0.3, 0.3], rounded to 1 decimal place)

Window around index 32075 (after interpolation):


,time,temperature_2m (°C),relative_humidity_2m (%),precipitation (mm),shortwave_radiation (W/m²),direct_radiation (W/m²),sunshine_duration (s),wind_speed_10m (km/h),weather_code (wmo code),dew_point_2m (°C)
32070,2023-12-01 01:30:00,1.7,95,0.0,0.0,0,0.0,3.2,45,1.0
32071,2023-12-01 01:45:00,1.6,95,0.0,0.0,0,0.0,3.4,45,0.9
32072,2023-12-01 02:00:00,1.6,95,0.0,0.0,0,0.0,3.4,45,0.9
32073,2023-12-01 02:15:00,1.6,95,0.0,0.0,0,0.0,3.2,61,0.9
32074,2023-12-01 02:30:00,1.6,94,0.0,0.0,0,0.0,3.3,61,0.8
32075,2023-12-01 02:45:00,1.6,94,0.1,0.0,0,0.0,3.1,61,0.8
32076,2023-12-01 03:00:00,1.5,94,0.2,0.0,0,0.0,3.2,61,0.7
32077,2023-12-01 03:15:00,1.3,95,0.1,0.0,0,0.0,3.2,71,0.6
32078,2023-12-01 03:30:00,1.0,96,0.2,0.0,0,0.0,2.9,71,0.4
32079,2023-12-01 03:45:00,0.6,97,0.3,0.0,0,0.0,2.8,71,0.2


## 6. Duplicate Rows & Timestamp Integrity (DST Checks on Individual Datasets)
- **SMARD (`Consumption` & `Prices`)**: Local `Europe/Berlin` wall-clock timestamps exhibit 4 repeated quarter-hours each October fallback (`oct_dst_fallback_rows`) and 4 skipped quarter-hours each March spring-forward (`march_dst_skipped_slots`).
- **Open-Meteo (`Weather`)**: Fixed `UTC+2` (`utc_offset_seconds = 7200`) has no DST jumps or repeated timestamps.

In [41]:
dup_summary_rows = []

for name, df in coerced_datasets.items():
    _, primary_ts = timestamp_config[name]
    ts = df[primary_ts]
    valid_ts = ts.dropna()
    full_dups = int(df.duplicated().sum())
    ts_dups = int(ts.duplicated().sum())

    dst_dup_rows = 0
    non_dst_dup_rows = 0
    if ts_dups > 0:
        dup_timestamps = valid_ts[valid_ts.duplicated(keep=False)]
        oct_dst_mask = (
            (dup_timestamps.dt.month == 10)
            & (dup_timestamps.dt.weekday == 6)
            & (dup_timestamps.dt.hour == 2)
        )
        dst_dup_rows = int(oct_dst_mask.sum())
        non_dst_dup_rows = int((~oct_dst_mask).sum())

    expected_grid = pd.date_range(start=valid_ts.min(), end=valid_ts.max(), freq="15min")
    missing_wall_clock = expected_grid.difference(valid_ts.unique())
    march_dst_mask = (
        (missing_wall_clock.month == 3)
        & (missing_wall_clock.weekday == 6)
        & (missing_wall_clock.hour == 2)
    )

    dup_summary_rows.append({
        "dataset": name,
        "start_ts": valid_ts.min(),
        "end_ts": valid_ts.max(),
        "full_row_duplicates": full_dups,
        "duplicate_timestamps": ts_dups,
        "oct_dst_fallback_rows": dst_dup_rows,
        "unexpected_duplicate_rows": non_dst_dup_rows,
        "march_dst_skipped_slots": int(march_dst_mask.sum()),
        "unexpected_missing_slots": int((~march_dst_mask).sum()),
    })

display(pd.DataFrame(dup_summary_rows))

,dataset,start_ts,end_ts,full_row_duplicates,duplicate_timestamps,oct_dst_fallback_rows,unexpected_duplicate_rows,march_dst_skipped_slots,unexpected_missing_slots
0,Actual Consumption,2023-01-01,2024-12-31 23:45:00,0,8,16,0,8,0
1,Day-Ahead Prices,2023-01-01,2024-12-31 23:45:00,0,8,16,0,8,0
2,Open-Meteo Weather,2023-01-01,2024-12-31 23:45:00,0,0,0,0,0,0


## 7. Target Variable Summary (`grid load [MWh]` in Actual Consumption 2023–2024)

In [42]:
cons_coerced = coerced_datasets["Actual Consumption"]
target = cons_coerced[TARGET_COL].dropna()
q01, q05, q25, q50, q75, q95, q99 = target.quantile([0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99])

overall_stats = pd.DataFrame({
    "Metric": [
        "Count", "Missing", "Mean [MWh]", "Std [MWh]", "Min [MWh]",
        "1% Quantile [MWh]", "5% Quantile [MWh]", "25% Quantile [MWh]",
        "Median (50%) [MWh]", "75% Quantile [MWh]", "95% Quantile [MWh]",
        "99% Quantile [MWh]", "Max [MWh]", "IQR [MWh]", "Skewness", "Kurtosis",
    ],
    "Value": [
        f"{len(target):,}", f"{int(cons_coerced[TARGET_COL].isna().sum()):,}",
        f"{target.mean():.2f}", f"{target.std():.2f}", f"{target.min():.2f}",
        f"{q01:.2f}", f"{q05:.2f}", f"{q25:.2f}", f"{q50:.2f}", f"{q75:.2f}",
        f"{q95:.2f}", f"{q99:.2f}", f"{target.max():.2f}", f"{(q75 - q25):.2f}",
        f"{target.skew():.3f}", f"{target.kurtosis():.3f}",
    ],
})
print("Overall Target Distribution (2023–2024):")
display(overall_stats)

df_temp = cons_coerced[["Start date", TARGET_COL]].dropna(subset=[TARGET_COL]).copy()
df_temp["year"] = df_temp["Start date"].dt.year

print("Target Summary by Year (2023–2024):")
display(
    df_temp.groupby("year")[TARGET_COL]
    .agg(count="count", mean="mean", std="std", min="min", median="median", max="max")
    .round(2)
)

Overall Target Distribution (2023–2024):


,Metric,Value
0,Count,"70,176"
1,Missing,0
2,Mean [MWh],1624.89
3,Std [MWh],347.35
4,Min [MWh],872.75
5,1% Quantile [MWh],988.50
6,5% Quantile [MWh],1106.19
7,25% Quantile [MWh],1329.50
8,Median (50%) [MWh],1604.50
9,75% Quantile [MWh],1906.81


Target Summary by Year (2023–2024):


,count,mean,std,min,median,max
year,,,,,,
2023,35040,1628.14,352.10,872.75,1612.00,2459.75
2024,35136,1621.64,342.53,907.00,1597.12,2498.00
